# E-Commerce Data Warehouse Pipeline

This notebook implements a complete data warehouse pipeline for the Udacity Data Engineering course.

**Business Context:** You are a data engineer at a fast-growing e-commerce company. Critical data is spread across multiple operational systems (PostgreSQL, Cassandra, Neo4j), making it difficult for analysts to run consistent reports. Your job is to design and implement a centralized analytics warehouse in Amazon Redshift.

## Tasks Overview

1. **Explore and Plan** - Review CSV data, identify key fields, map to warehouse schema
2. **Design the Schema** - Create dimensional model with staging, dimension, and fact tables
3. **Extract and Transform** - Load source systems and extract/transform data
4. **Load into Redshift** - Execute DDL, load staging tables, populate dimensions and facts
5. **Optimize Performance** - Apply best practices, create materialized views
6. **Validate and Report** - Run quality checks, generate final report

---
## The Data

The dataset consists of three CSV files representing data from different operational systems:

### Orders Data (PostgreSQL source) - `ecom_orders_postgres.csv`
- **order_id**: Unique identifier for each order
- **customer_id**: Customer who placed the order
- **order_datetime / ship_datetime**: Timestamps for order and shipping
- **channel / device_type / browser**: How the order was placed
- **country / state**: Geographic location
- **payment_method / campaign**: Payment and marketing info
- **Financial fields**: subtotal, discount, shipping, tax, total amounts
- **Delivery fields**: delivery_days, on_time_delivery
- **Flags**: authorization_approved, returned

### Events Data (Cassandra source) - `ecom_events_cassandra.csv`
- **event_id / session_id**: Event and session identifiers
- **customer_id**: Customer who triggered the event
- **event_type**: Type of event (page_view, product_view, add_to_cart, etc.)
- **event_ts**: Timestamp of the event
- **Device/browser/OS info**: Technical context
- **Behavioral fields**: page_depth, latency_ms, dwell_seconds
- **Commerce fields**: cart_value_usd, discount_rate, fraud_score

### Graph Edges Data (Neo4j source) - `ecom_graph_edges_neo4j.csv`
- **edge_id**: Unique relationship identifier
- **from_node_id / to_node_id**: Source and target nodes
- **from_node_type / to_node_type**: Node types (Customer, Product, Order)
- **relationship**: Type of relationship (PURCHASED, VIEWED, ADDED_TO_CART, etc.)
- **Context fields**: order_id, category, customer_segment, campaign
- **Metrics**: edge_strength, unit_price_usd, quantity

---
## Setup: Imports and Dependencies

Run this cell first to import all required libraries.

In [1]:
# ========= Imports
import os, io, re, time, json, textwrap
from datetime import datetime
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd

# Source system libraries
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from neo4j import GraphDatabase

# Warehouse (Redshift) via Data API
import boto3

# Optional progress bars
try:
    from tqdm import tqdm
    TQDM = True
except Exception:
    TQDM = False

print("All imports successful!")
print(f"   - pandas version: {pd.__version__}")
print(f"   - numpy version: {np.__version__}")

All imports successful!
   - pandas version: 3.0.6
   - numpy version: 2.5.3


---
## Setup: Configuration

Update these settings for your environment. You will need to:
1. Set your AWS credentials (from Cloud Resources)
2. Configure database connection parameters

In [2]:
# Credentials and connection settings are read from .env (see .env.example).
from dotenv import load_dotenv
load_dotenv(override=True)


True

In [3]:
# ========= Configuration
BASE_DIR = os.getenv("PROJECT_BASE_DIR", ".")
DATA_DIR = os.path.join(BASE_DIR, "data")
CSV_ORDERS  = os.path.join(DATA_DIR, "ecom_orders_postgres.csv")
CSV_EVENTS  = os.path.join(DATA_DIR, "ecom_events_cassandra.csv")
CSV_EDGES   = os.path.join(DATA_DIR, "ecom_graph_edges_neo4j.csv")
DDL_MD_PATH = os.path.join(BASE_DIR, "project-ddl-long.md")
MERMAID_MD  = os.path.join(BASE_DIR, "project-mermaid-diagram.md")
BATCH_SIZE  = int(os.getenv("BATCH_SIZE", "1000"))

# PostgreSQL
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB",   "postgres")
PG_USER = os.getenv("PG_USER", "temp")
PG_PW   = os.getenv("PG_PW",   "temp")

# Cassandra
CAS_HOSTS = os.getenv("CAS_HOSTS", "localhost").split(",")
CAS_PORT  = int(os.getenv("CAS_PORT", "9042"))
CAS_USER  = os.getenv("CAS_USER", "")
CAS_PW    = os.getenv("CAS_PW", "")
CAS_KEYSPACE = os.getenv("CAS_KEYSPACE", "ecommerce")

# Neo4j
NEO4J_URI  = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PW   = os.getenv("NEO4J_PW",   "neo4jpass")

# AWS/Redshift
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
REDSHIFT_DATABASE = os.getenv("REDSHIFT_DATABASE", "ecom")
REDSHIFT_WORKGROUP = os.getenv("REDSHIFT_WORKGROUP", "udacity-dwh-wg")
REDSHIFT_SECRET_ARN = os.getenv("REDSHIFT_SECRET_ARN")
REDSHIFT_CLUSTER_IDENTIFIER = os.getenv("REDSHIFT_CLUSTER_IDENTIFIER")
REDSHIFT_DB_USER = os.getenv("REDSHIFT_DB_USER")

# Verify configuration
print("Configuration loaded!")
print(f"   - BASE_DIR: {BASE_DIR}")
print(f"   - PostgreSQL: {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"   - Cassandra: {CAS_HOSTS}:{CAS_PORT}/{CAS_KEYSPACE}")
print(f"   - Neo4j: {NEO4J_URI}")
print(f"   - Redshift: {REDSHIFT_DATABASE} (workgroup: {REDSHIFT_WORKGROUP})")

print()
if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    print(f"   AWS credentials found (Key ID: {AWS_ACCESS_KEY_ID[:10]}...)")
else:
    print("   WARNING: AWS credentials NOT FOUND - set them above!")

Configuration loaded!
   - BASE_DIR: .
   - PostgreSQL: localhost:5432/postgres
   - Cassandra: ['localhost']:9042/ecommerce
   - Neo4j: bolt://localhost:7687
   - Redshift: ecom (workgroup: udacity-dwh-wg)

   AWS credentials found (Key ID: ASIAQ6JP6R...)


---
## Setup: Column Specifications

These define the mapping from source columns to Redshift staging tables.
Use these as a reference when building your transformation logic.

In [4]:
# Column specs for Redshift staging (name, kind)
# kind: 's' = string, 'ts' = timestamp, 'i' = integer, 'f' = float, 'b' = boolean

ORDERS_COLSPEC = [
    ('order_id','s'),('customer_id','s'),('order_datetime','ts'),('ship_datetime','ts'),
    ('channel','s'),('device_type','s'),('browser','s'),('country','s'),('state','s'),
    ('payment_method','s'),('campaign','s'),('primary_category','s'),('num_distinct_items','i'),
    ('subtotal_usd','f'),('discount_rate','f'),('discount_amount_usd','f'),('shipping_method','s'),
    ('shipping_cost_usd','f'),('tax_rate','f'),('tax_amount_usd','f'),('order_total_usd','f'),
    ('order_weight_kg','f'),('delivery_days','i'),('on_time_delivery','b'),
    ('authorization_approved','b'),('returned','b')
]

EVENTS_COLSPEC = [
    ('event_id','s'),('customer_id','s'),('session_id','s'),('event_type','s'),('event_ts','ts'),
    ('device_type','s'),('browser','s'),('os','s'),('referrer','s'),('country','s'),('state','s'),
    ('ab_variant','s'),('is_logged_in','b'),('page_depth','i'),('latency_ms','i'),
    ('dwell_seconds','i'),('cart_value_usd','f'),('discount_rate','f'),('fraud_score','f'),
    ('payment_outcome','s'),('sequence_num','i'),('product_id','s'),('category','s'),('promo_code','s')
]

EDGES_COLSPEC = [
    ('edge_id','s'),('from_node_id','s'),('from_node_type','s'),('to_node_id','s'),('to_node_type','s'),
    ('relationship','s'),('timestamp','ts'),('order_id','s'),('category','s'),('customer_segment','s'),
    ('edge_strength','f'),('price_bucket','s'),('region','s'),('state','s'),('campaign','s'),
    ('same_household','b'),('prior_interactions','i'),('dwell_seconds','i'),('product_id','s'),
    ('unit_price_usd','f'),('quantity','i'),('returned_flag','b'),('auth_approved','b')
]

print(f"Column specs defined:")
print(f"   - Orders: {len(ORDERS_COLSPEC)} columns")
print(f"   - Events: {len(EVENTS_COLSPEC)} columns")
print(f"   - Edges: {len(EDGES_COLSPEC)} columns")

Column specs defined:
   - Orders: 26 columns
   - Events: 24 columns
   - Edges: 23 columns


---
## Setup: Helper Functions

Utility functions used throughout the pipeline.

In [5]:
def trim_df(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize text fields and handle NaN values."""
    df = df.copy()
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({'nan': np.nan, 'None': np.nan, 'NaN': np.nan, '': np.nan})
    return df

def read_csvs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Read all three source CSVs and apply cleaning."""
    orders = pd.read_csv(CSV_ORDERS)
    events = pd.read_csv(CSV_EVENTS)
    edges  = pd.read_csv(CSV_EDGES)
    return trim_df(orders), trim_df(events), trim_df(edges)

print("Helper functions defined: trim_df(), read_csvs()")

Helper functions defined: trim_df(), read_csvs()


---
# Task 1: Explore and Plan the Data Pipeline

In this task, you will:
- Review the provided CSV data files to understand structure, columns, and content
- Identify key fields and relationships important for analysis
- Map source fields to fact and dimension tables
- Consider data format standardization needs

**Deliverables:**
- Written plan mapping fields from all three sources to fact and dimension tables
- Documentation of key relationships and ID standardization strategies

In [6]:
# Read the CSV files
print("Reading CSV files...")
orders_df, events_df, edges_df = read_csvs()

print("\n" + "="*60)
print("TASK 1: Data Exploration")
print("="*60)

# Explore the orders data
print(f"\n📊 ORDERS DATA (from PostgreSQL)")
print(f"   Shape: {orders_df.shape[0]} rows, {orders_df.shape[1]} columns")
print(f"   Columns: {list(orders_df.columns)}")
print("\n   Dtypes:")
print(orders_df.dtypes.to_string())
display(orders_df.head())


Reading CSV files...

TASK 1: Data Exploration

📊 ORDERS DATA (from PostgreSQL)
   Shape: 2500 rows, 26 columns
   Columns: ['order_id', 'customer_id', 'order_datetime', 'ship_datetime', 'channel', 'device_type', 'browser', 'country', 'state', 'payment_method', 'campaign', 'primary_category', 'num_distinct_items', 'subtotal_usd', 'discount_rate', 'discount_amount_usd', 'shipping_method', 'shipping_cost_usd', 'tax_rate', 'tax_amount_usd', 'order_total_usd', 'order_weight_kg', 'delivery_days', 'on_time_delivery', 'authorization_approved', 'returned']

   Dtypes:
order_id                      str
customer_id                   str
order_datetime                str
ship_datetime                 str
channel                       str
device_type                   str
browser                       str
country                       str
state                         str
payment_method                str
campaign                      str
primary_category              str
num_distinct_items       

/var/folders/sh/tl74dmfn2d38g0d22l4xvvhw0000gn/T/ipykernel_58697/2325207208.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include=['object']).columns:
/var/folders/sh/tl74dmfn2d38g0d22l4xvvhw0000gn/T/ipykernel_58697/2325207208.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pand

,order_id,customer_id,order_datetime,ship_datetime,channel,device_type,browser,country,state,payment_method,...,shipping_method,shipping_cost_usd,tax_rate,tax_amount_usd,order_total_usd,order_weight_kg,delivery_days,on_time_delivery,authorization_approved,returned
0,ORD100000,C29457,2024-06-12 02:14:21,2024-06-17 02:14:21,android_app,mobile,Safari,CA,CT,google_pay,...,standard,5.05,0.0000,0.00,838.39,0.10,5,True,True,False
1,ORD100001,C22666,2024-08-08 00:16:41,2024-08-11 00:16:41,mobile_web,desktop,Opera,DE,OH,apple_pay,...,standard,9.35,0.0000,0.00,388.60,0.20,3,True,True,False
2,ORD100002,C72623,2024-09-11 06:59:18,2024-09-15 06:59:18,web,desktop,Edge,US,TX,google_pay,...,standard,6.63,0.0923,59.32,708.60,0.85,4,True,True,False
3,ORD100003,C62733,2025-03-13 15:54:14,2025-03-17 15:54:14,web,mobile,Safari,UK,VT,google_pay,...,standard,7.15,0.0000,0.00,104.23,0.10,4,True,True,False
4,ORD100004,C62083,2025-02-28 00:20:21,2025-03-06 00:20:21,android_app,tablet,Opera,US,TN,apple_pay,...,standard,3.96,0.0864,3.69,50.41,0.29,6,False,True,False


In [7]:
# Explore the events data
print(f"\n📊 EVENTS DATA (from Cassandra)")
print(f"   Shape: {events_df.shape[0]} rows, {events_df.shape[1]} columns")
print(f"   Columns: {list(events_df.columns)}")
print("\n   Event types:")
print(events_df["event_type"].value_counts().to_string())
display(events_df.head())



📊 EVENTS DATA (from Cassandra)
   Shape: 2500 rows, 24 columns
   Columns: ['event_id', 'customer_id', 'session_id', 'event_type', 'event_ts', 'device_type', 'browser', 'os', 'referrer', 'country', 'state', 'ab_variant', 'is_logged_in', 'page_depth', 'latency_ms', 'dwell_seconds', 'cart_value_usd', 'discount_rate', 'fraud_score', 'payment_outcome', 'sequence_num', 'product_id', 'category', 'promo_code']

   Event types:
event_type
product_view        696
page_view           641
add_to_cart         480
checkout_start      259
payment_attempt     218
purchase            155
return_initiated     51


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200000,C47857,S7645111197,page_view,2024-11-14 00:17:09,desktop,Firefox,Linux,organic_search,IN,...,331,35,18.12,0.165,0.000,NaN,3,NaN,NaN,NONE
1,EVT200001,C83195,S1423573902,page_view,2024-08-15 10:42:18,mobile,Chrome,Linux,social,DE,...,432,26,33.62,0.083,0.262,NaN,2,NaN,NaN,NONE
2,EVT200002,C27664,S2829853742,product_view,2024-05-03 23:28:24,mobile,Chrome,Android,email,BR,...,360,6,12.26,0.237,0.162,NaN,5,P5735,Home,FREESHIP
3,EVT200003,C55911,S2500554740,add_to_cart,2024-10-28 18:44:50,desktop,Firefox,Linux,direct,US,...,414,22,86.55,0.197,0.067,NaN,1,P1730,Pets,FREESHIP
4,EVT200004,C27347,S3072616664,add_to_cart,2024-07-22 07:43:54,mobile,Chrome,Linux,direct,CA,...,381,32,87.78,0.058,0.000,NaN,8,P6252,Pets,WELCOME10


In [8]:
# Explore the graph edges data
print(f"\n📊 GRAPH EDGES DATA (from Neo4j)")
print(f"   Shape: {edges_df.shape[0]} rows, {edges_df.shape[1]} columns")
print(f"   Columns: {list(edges_df.columns)}")
print("\n   Relationships by node types:")
print(edges_df.groupby(["from_node_type", "relationship", "to_node_type"]).size().to_string())
display(edges_df.head())



📊 GRAPH EDGES DATA (from Neo4j)
   Shape: 2500 rows, 23 columns
   Columns: ['edge_id', 'from_node_id', 'from_node_type', 'to_node_id', 'to_node_type', 'relationship', 'timestamp', 'order_id', 'category', 'customer_segment', 'edge_strength', 'price_bucket', 'region', 'state', 'campaign', 'same_household', 'prior_interactions', 'dwell_seconds', 'product_id', 'unit_price_usd', 'quantity', 'returned_flag', 'auth_approved']

   Relationships by node types:
from_node_type  relationship      to_node_type
Customer        ADDED_TO_CART     Product         441
                PURCHASED         Product         469
                REFERRED_FRIEND   Customer        256
                RETURNS           Product         111
                VIEWED            Product         830
Product         ALSO_BOUGHT_WITH  Product         393


,edge_id,from_node_id,from_node_type,to_node_id,to_node_type,relationship,timestamp,order_id,category,customer_segment,...,state,campaign,same_household,prior_interactions,dwell_seconds,product_id,unit_price_usd,quantity,returned_flag,auth_approved
0,EDGE300000,C65482,Customer,P2261,Product,RETURNS,2024-01-27 13:42:26,ORD100938,Books,New,...,WA,SpringSale,False,3,66,P2261,315.87,2,False,False
1,EDGE300001,C59599,Customer,P7625,Product,PURCHASED,2025-02-07 10:22:55,ORD102431,Toys,Loyal,...,WI,Holiday,False,4,62,P7625,19.21,1,False,True
2,EDGE300002,C44719,Customer,P8677,Product,ADDED_TO_CART,2024-01-06 14:04:37,NaN,Pets,Active,...,KS,Loyalty,False,5,16,P8677,68.68,2,False,False
3,EDGE300003,C13897,Customer,P7883,Product,VIEWED,2024-06-04 21:36:14,NaN,Grocery,Active,...,NV,Clearance,False,3,44,P7883,74.70,1,False,False
4,EDGE300004,C30350,Customer,P2015,Product,ADDED_TO_CART,2024-06-23 20:00:17,NaN,Toys,Active,...,AR,SpringSale,False,2,17,P2015,52.71,1,False,False


In [15]:
edges_df

,edge_id,from_node_id,from_node_type,to_node_id,to_node_type,relationship,timestamp,order_id,category,customer_segment,...,state,campaign,same_household,prior_interactions,dwell_seconds,product_id,unit_price_usd,quantity,returned_flag,auth_approved
0,EDGE300000,C65482,Customer,P2261,Product,RETURNS,2024-01-27 13:42:26,ORD100938,Books,New,...,WA,SpringSale,False,3,66,P2261,315.87,2,False,False
1,EDGE300001,C59599,Customer,P7625,Product,PURCHASED,2025-02-07 10:22:55,ORD102431,Toys,Loyal,...,WI,Holiday,False,4,62,P7625,19.21,1,False,True
2,EDGE300002,C44719,Customer,P8677,Product,ADDED_TO_CART,2024-01-06 14:04:37,NaN,Pets,Active,...,KS,Loyalty,False,5,16,P8677,68.68,2,False,False
3,EDGE300003,C13897,Customer,P7883,Product,VIEWED,2024-06-04 21:36:14,NaN,Grocery,Active,...,NV,Clearance,False,3,44,P7883,74.70,1,False,False
4,EDGE300004,C30350,Customer,P2015,Product,ADDED_TO_CART,2024-06-23 20:00:17,NaN,Toys,Active,...,AR,SpringSale,False,2,17,P2015,52.71,1,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,EDGE302495,P2953,Product,P8836,Product,ALSO_BOUGHT_WITH,2024-03-24 04:55:46,ORD100697,Apparel,New,...,OH,Loyalty,False,1,42,P2953,99.93,1,False,False
2496,EDGE302496,C36318,Customer,P7882,Product,VIEWED,2024-09-12 06:55:43,NaN,Toys,Active,...,WI,Loyalty,False,3,59,P7882,630.61,2,False,False
2497,EDGE302497,C54023,Customer,P6254,Product,PURCHASED,2025-03-22 14:19:23,ORD100873,Grocery,Active,...,NV,SpringSale,False,4,27,P6254,36.77,1,False,True
2498,EDGE302498,C11986,Customer,P8946,Product,RETURNS,2025-04-15 10:45:09,ORD101864,Sports,AtRisk,...,VA,NewArrivals,False,3,31,P8946,64.26,2,False,False


In [9]:
# Identify key fields and relationships
print("\n" + "="*60)
print("KEY FIELDS AND RELATIONSHIPS")
print("="*60)

SOURCES = [
    ("orders", orders_df, ORDERS_COLSPEC, "order_id"),
    ("events", events_df, EVENTS_COLSPEC, "event_id"),
    ("edges",  edges_df,  EDGES_COLSPEC,  "edge_id"),
]

print("\n Primary Keys:")
for name, df, _, key in SOURCES:
    row_count = len(df)
    unique_count = df[key].nunique()
    status = "unique" if unique_count == row_count else "DUPLICATES"
    print(f"   {name}.{key}: {unique_count:,} unique of {row_count:,} rows -> {status}")

print("\n Foreign Key Relationships:")
cust_orders = set(orders_df.customer_id.dropna())
cust_events = set(events_df.customer_id.dropna())
cust_edges = (set(edges_df.loc[edges_df.from_node_type == "Customer", "from_node_id"])
              | set(edges_df.loc[edges_df.to_node_type == "Customer", "to_node_id"]))
print(f"   customers: orders={len(cust_orders):,}, events={len(cust_events):,}, edges={len(cust_edges):,}")
print(f"   orders ∩ events: {len(cust_orders & cust_events):,}   orders ∩ edges: {len(cust_orders & cust_edges):,}   "
      f"all sources (union): {len(cust_orders | cust_events | cust_edges):,}")
prod_events = set(events_df.product_id.dropna())
prod_edges = (set(edges_df.product_id.dropna())
              | set(edges_df.loc[edges_df.from_node_type == "Product", "from_node_id"])
              | set(edges_df.loc[edges_df.to_node_type == "Product", "to_node_id"]))
print(f"   products: events={len(prod_events):,}, edges={len(prod_edges):,}, overlap={len(prod_events & prod_edges):,}")
edge_orders = set(edges_df.order_id.dropna())
print(f"   edges.order_id found in orders: {len(edge_orders & set(orders_df.order_id)):,} of {len(edge_orders):,}")
sess = events_df.session_id.nunique()
print(f"   events: {sess:,} sessions across {len(events_df):,} events")

print("\n Date/Time Fields:")
for source_name, source_df, colspec, _primary_key in SOURCES:
    for column, kind in colspec:
        if kind == "ts":
            timestamps = pd.to_datetime(source_df[column])
            print(f"   {source_name}.{column}: {timestamps.min()} -> {timestamps.max()}  (nulls: {timestamps.isna().sum()})")

print("\n Categorical Fields (potential dimensions):")
for source_name, source_df, colspec, _primary_key in SOURCES:
    categorical_cardinality = {
        column: source_df[column].nunique()
        for column, kind in colspec
        if kind == "s" and not column.endswith("_id") and source_df[column].nunique() <= 50
    }
    print(f"   {source_name}: {categorical_cardinality}")


KEY FIELDS AND RELATIONSHIPS

 Primary Keys:
   orders.order_id: 2,500 unique of 2,500 rows -> unique
   events.event_id: 2,500 unique of 2,500 rows -> unique
   edges.edge_id: 2,500 unique of 2,500 rows -> unique

 Foreign Key Relationships:
   customers: orders=2,460, events=2,461, edges=2,340
   orders ∩ events: 72   orders ∩ edges: 72   all sources (union): 7,057
   products: events=1,276, edges=2,462, overlap=378
   edges.order_id found in orders: 796 of 796
   events: 2,500 sessions across 2,500 events

 Date/Time Fields:
   orders.order_datetime: 2024-01-01 01:48:15 -> 2025-06-29 15:18:02  (nulls: 0)
   orders.ship_datetime: 2024-01-03 21:57:58 -> 2025-07-06 04:31:14  (nulls: 0)
   events.event_ts: 2024-01-01 09:05:14 -> 2025-06-29 19:22:01  (nulls: 0)
   edges.timestamp: 2024-01-01 01:19:07 -> 2025-06-29 22:30:45  (nulls: 0)

 Categorical Fields (potential dimensions):
   orders: {'channel': 5, 'device_type': 3, 'browser': 5, 'country': 6, 'payment_method': 5, 'campaign': 6, '

In [10]:
# Document your field mappings and data quality observations
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

# Null values per source (only columns that have any)
for name, df, _, key in SOURCES:
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    print(f"\n{name}: {len(df):,} rows, null key ({key}) = {df[key].isna().sum()}, duplicate keys = {df[key].duplicated().sum()}")
    if nulls.empty:
        print("   no null values")
    else:
        print((nulls.to_frame("nulls").assign(pct=lambda x: (x.nulls / len(df) * 100).round(1))).to_string())

# Business-rule checks on orders
calc = orders_df.subtotal_usd - orders_df.discount_amount_usd + orders_df.shipping_cost_usd + orders_df.tax_amount_usd
print(f"\norders where total != subtotal - discount + shipping + tax: {((calc - orders_df.order_total_usd).abs() > 0.01).sum():,}")
print(f"orders shipped before ordered: {(pd.to_datetime(orders_df.ship_datetime) < pd.to_datetime(orders_df.order_datetime)).sum():,}")
print(f"events with negative latency/dwell: {((events_df.latency_ms < 0) | (events_df.dwell_seconds < 0)).sum():,}")
print(f"edges with null from/to node id: {edges_df[['from_node_id', 'to_node_id']].isna().any(axis=1).sum():,}")



DATA QUALITY SUMMARY

orders: 2,500 rows, null key (order_id) = 0, duplicate keys = 0
          nulls   pct
campaign    409  16.4

events: 2,500 rows, null key (event_id) = 0, duplicate keys = 0
                 nulls   pct
state             2210  88.4
payment_outcome   2127  85.1
product_id        1118  44.7
category          1118  44.7

edges: 2,500 rows, null key (edge_id) = 0, duplicate keys = 0
          nulls   pct
order_id   1527  61.1
region      652  26.1
campaign    352  14.1

orders where total != subtotal - discount + shipping + tax: 0
orders shipped before ordered: 0
events with negative latency/dwell: 0
edges with null from/to node id: 0


---
# Task 2: Design the Warehouse Schema

In this task, you will:
- Review the dimensional (star) schema design
- Understand staging tables, dimension tables, and fact tables
- Review distribution keys, sort keys, and encoding for optimization
- Document the purpose of each table

The DDL is defined in `project-ddl-long.md`. Review the schema design and understand how it supports analytics.

**Deliverables:**
- Understanding of the provided DDL structure
- Documentation of table purposes and query support

In [12]:
# Review the DDL file
print("="*60)
print("TASK 2: Warehouse Schema Design")
print("="*60)

print("\n📄 Reading DDL file:", DDL_MD_PATH)

# Read and parse the DDL file
with open(DDL_MD_PATH, 'r') as f:
    ddl_content = f.read()

sql_blocks = re.findall(r"```sql(.*?)```", ddl_content, flags=re.S)
print(f"Found {len(sql_blocks)} SQL blocks")

# Count and list the tables defined
table_names = re.findall(
    r"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?(\w+\.\w+)",
    "\n".join(sql_blocks),
    flags=re.I,
)

staging_tables = [t for t in table_names if t.startswith("stg.")]
dim_tables = [t for t in table_names if t.startswith("dw.dim_")]
fact_tables = [t for t in table_names if t.startswith("dw.fact_")]

for label, tables in [("Staging", staging_tables), ("Dimension", dim_tables), ("Fact", fact_tables)]:
    print(f"\n{label} tables ({len(tables)}):")
    for t in tables:
        print("  -", t)

TASK 2: Warehouse Schema Design

📄 Reading DDL file: ./project-ddl-long.md
Found 5 SQL blocks

Staging tables (3):
  - stg.orders_raw
  - stg.events_raw
  - stg.edges_raw

Dimension tables (12):
  - dw.dim_date
  - dw.dim_customer
  - dw.dim_product
  - dw.dim_campaign
  - dw.dim_channel
  - dw.dim_device
  - dw.dim_browser
  - dw.dim_os
  - dw.dim_referrer
  - dw.dim_shipping_method
  - dw.dim_payment_method
  - dw.dim_ab_variant

Fact tables (3):
  - dw.fact_orders
  - dw.fact_events
  - dw.fact_graph_edges


In [13]:
# Document the schema design
print("\n" + "="*60)
print("SCHEMA DESIGN DOCUMENTATION")
print("="*60)

schema_doc = """
## Staging Tables (stg schema)
Permissive landing zone: wide VARCHARs, minimal constraints, one table per source.
Staging decouples extraction from modelling, so raw data can be loaded, inspected and
reloaded without touching the warehouse tables.
- stg.orders_raw  : orders from PostgreSQL
- stg.events_raw  : clickstream events from Cassandra
- stg.edges_raw   : customer/product relationships from Neo4j

## Dimension Tables (dw schema)
Surrogate keys (IDENTITY) are used as the join keys from the facts.
- dim_customer, dim_product : SCD Type 2 ready (effective_from/to, is_current);
                              fact loads join on is_current = TRUE
- dim_date                  : yyyymmdd integer key; used for order, ship and event dates
- Small lookups             : dim_campaign, dim_channel, dim_device, dim_browser, dim_os,
                              dim_referrer, dim_shipping_method, dim_payment_method,
                              dim_ab_variant

## Fact Tables (dw schema)
- fact_orders      : one row per order (order_id); customer, order/ship date, channel,
                     device, browser, campaign, payment and shipping method keys
- fact_events      : one row per event (event_id); customer, product, date, channel,
                     device, browser, os, referrer, ab_variant keys; session_id and
                     event_type kept on the fact as degenerate dimensions
- fact_graph_edges : one row per graph edge (edge_id); from/to customer and product keys,
                     date and campaign keys

## Optimization Choices
- DISTKEY(customer_sk) on fact_orders and fact_events: rows for the same customer are
  collocated, so customer-centric joins and aggregations avoid data movement
- DISTKEY(to_product_sk) on fact_graph_edges: supports product-centric relationship analysis
- DISTKEY on business key (customer_id, product_id) for the large dimensions
- DISTSTYLE ALL on small lookup dimensions and dim_date: a full copy on every node, so
  joins to them are always local
- SORTKEY on the date key for all facts: time-range filters skip unneeded blocks;
  dimensions are sorted on their business key
- ENCODE zstd on most columns: good compression and less I/O
"""

print(schema_doc)


SCHEMA DESIGN DOCUMENTATION

## Staging Tables (stg schema)
Permissive landing zone: wide VARCHARs, minimal constraints, one table per source.
Staging decouples extraction from modelling, so raw data can be loaded, inspected and
reloaded without touching the warehouse tables.
- stg.orders_raw  : orders from PostgreSQL
- stg.events_raw  : clickstream events from Cassandra
- stg.edges_raw   : customer/product relationships from Neo4j

## Dimension Tables (dw schema)
Surrogate keys (IDENTITY) are used as the join keys from the facts.
- dim_customer, dim_product : SCD Type 2 ready (effective_from/to, is_current);
                              fact loads join on is_current = TRUE
- dim_date                  : yyyymmdd integer key; used for order, ship and event dates
- Small lookups             : dim_campaign, dim_channel, dim_device, dim_browser, dim_os,
                              dim_referrer, dim_shipping_method, dim_payment_method,
                              dim_ab_variant

## Fa

---
# Task 3: Extract and Transform the Source Data

In this task, you will:
- Load CSV data into PostgreSQL, Cassandra, and Neo4j (simulating production)
- Write extraction functions to query each source system
- Apply transformations to clean and conform the data
- Ensure transformed data matches staging table specifications

**Deliverables:**
- Working functions to connect, load, and extract from each source
- Transformed DataFrames ready for Redshift loading

## Task 3.1: Define Source System Functions

Implement the connection and data loading functions for each source system.

In [14]:
# ========= PostgreSQL Functions =========

def pg_connect():
    """Connect to PostgreSQL."""
    return psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PW)

PG_ORDERS_DDL = """
CREATE TABLE IF NOT EXISTS raw.orders (
  order_id VARCHAR(32) PRIMARY KEY, customer_id VARCHAR(32),
  order_datetime TIMESTAMP, ship_datetime TIMESTAMP,
  channel VARCHAR(32), device_type VARCHAR(16), browser VARCHAR(16),
  country VARCHAR(8), state VARCHAR(8), payment_method VARCHAR(16),
  campaign VARCHAR(32), primary_category VARCHAR(32), num_distinct_items INTEGER,
  subtotal_usd NUMERIC(12,2), discount_rate NUMERIC(5,3), discount_amount_usd NUMERIC(12,2),
  shipping_method VARCHAR(16), shipping_cost_usd NUMERIC(12,2),
  tax_rate NUMERIC(6,4), tax_amount_usd NUMERIC(12,2), order_total_usd NUMERIC(12,2),
  order_weight_kg NUMERIC(10,2), delivery_days INTEGER,
  on_time_delivery BOOLEAN, authorization_approved BOOLEAN, returned BOOLEAN
)"""

def _py(v):
    """Convert pandas/numpy scalars (incl. NaN/NaT) to plain Python for psycopg2."""
    if pd.isna(v):
        return None
    return v.item() if hasattr(v, "item") else (v.to_pydatetime() if hasattr(v, "to_pydatetime") else v)

def pg_load_orders(df: pd.DataFrame):
    """Create table and load orders data into PostgreSQL."""
    cols = [c for c, _ in ORDERS_COLSPEC]
    df = df.copy()
    for c, k in ORDERS_COLSPEC:
        if k == 'ts':
            df[c] = pd.to_datetime(df[c])
        elif k == 'b':
            df[c] = df[c].map({'True': True, 'False': False, True: True, False: False})
    rows = [tuple(_py(v) for v in r) for r in df[cols].itertuples(index=False, name=None)]
    with pg_connect() as conn, conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS raw;")
        cur.execute(PG_ORDERS_DDL)
        cur.execute("TRUNCATE raw.orders;")
        for i in range(0, len(rows), BATCH_SIZE):
            execute_values(cur, f"INSERT INTO raw.orders ({','.join(cols)}) VALUES %s", rows[i:i+BATCH_SIZE])
    print(f"   Loaded {len(rows)} orders into raw.orders")

def extract_from_pg() -> pd.DataFrame:
    """Extract orders from PostgreSQL."""
    engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PW}@{PG_HOST}:{PG_PORT}/{PG_DB}")
    return pd.read_sql_query("SELECT * FROM raw.orders", engine)

print("PostgreSQL functions defined")


PostgreSQL functions defined


In [15]:
# ========= Cassandra Functions =========
from cassandra.concurrent import execute_concurrent_with_args

def _records(df: pd.DataFrame, colspec) -> List[tuple]:
    """DataFrame -> list of plain-Python tuples (None for NaN), typed per colspec."""
    df = df.copy()
    out = []
    conv = {
        's': lambda v: str(v),
        'ts': lambda v: pd.Timestamp(v).to_pydatetime(),
        'i': lambda v: int(v),
        'f': lambda v: float(v),
        'b': lambda v: v if isinstance(v, bool) else str(v).strip().lower() == 'true',
    }
    fns = [conv[k] for _, k in colspec]
    for row in df[[c for c, _ in colspec]].itertuples(index=False, name=None):
        out.append(tuple(None if pd.isna(v) else f(v) for v, f in zip(row, fns)))
    return out

CAS_TYPES = {'s': 'text', 'ts': 'timestamp', 'i': 'int', 'f': 'double', 'b': 'boolean'}

def cas_connect():
    """Connect to Cassandra and ensure keyspace exists."""
    auth = PlainTextAuthProvider(CAS_USER, CAS_PW) if CAS_USER else None
    cluster = Cluster(CAS_HOSTS, port=CAS_PORT, auth_provider=auth)
    session = cluster.connect()
    session.execute(f"CREATE KEYSPACE IF NOT EXISTS {CAS_KEYSPACE} "
                    "WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}")
    session.set_keyspace(CAS_KEYSPACE)
    return session, cluster

def cas_load_events(df: pd.DataFrame):
    """Create table and load events data into Cassandra."""
    session, cluster = cas_connect()
    try:
        cols = ", ".join(f"{c} {CAS_TYPES[k]}" for c, k in EVENTS_COLSPEC)
        session.execute(f"CREATE TABLE IF NOT EXISTS events ({cols}, PRIMARY KEY (event_id))")
        session.execute("TRUNCATE events")
        names = [c for c, _ in EVENTS_COLSPEC]
        stmt = session.prepare(f"INSERT INTO events ({','.join(names)}) VALUES ({','.join('?' * len(names))})")
        results = execute_concurrent_with_args(session, stmt, _records(df, EVENTS_COLSPEC), concurrency=50)
        failed = [r for ok, r in results if not ok]
        if failed:
            raise RuntimeError(f"{len(failed)} inserts failed, e.g. {failed[0]}")
        print(f"   Loaded {len(results)} events into {CAS_KEYSPACE}.events")
    finally:
        cluster.shutdown()

def extract_from_cas() -> pd.DataFrame:
    """Extract events from Cassandra."""
    session, cluster = cas_connect()
    try:
        names = [c for c, _ in EVENTS_COLSPEC]
        rows = session.execute(f"SELECT {','.join(names)} FROM events")
        return pd.DataFrame(list(rows), columns=names)
    finally:
        cluster.shutdown()

print("Cassandra functions defined")


Cassandra functions defined


In [16]:
# ========= Neo4j Functions =========
NEO4J_REL_TYPES = {"VIEWED", "PURCHASED", "ADDED_TO_CART", "ALSO_BOUGHT_WITH", "REFERRED_FRIEND", "RETURNS"}
NEO4J_NODE_LABELS = {"Customer", "Product", "Order"}
EDGE_PROPS = [c for c, _ in EDGES_COLSPEC if c not in ("from_node_id", "from_node_type", "to_node_id", "to_node_type", "relationship")]

def neo4j_driver():
    """Connect to Neo4j."""
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PW))

def neo4j_load_edges(df: pd.DataFrame):
    """Load edges into Neo4j as nodes and relationships."""
    # Labels and relationship types can't be Cypher parameters, so whitelist then interpolate.
    bad = (set(df.relationship) - NEO4J_REL_TYPES) | ((set(df.from_node_type) | set(df.to_node_type)) - NEO4J_NODE_LABELS)
    if bad:
        raise ValueError(f"Unexpected relationship/label values: {bad}")
    df = df.dropna(subset=["edge_id", "from_node_id", "to_node_id"])
    recs = _records(df, EDGES_COLSPEC)
    names = [c for c, _ in EDGES_COLSPEC]
    dicts = []
    for r in recs:
        d = dict(zip(names, r))
        d["timestamp"] = d["timestamp"].isoformat() if d["timestamp"] else None
        dicts.append(d)
    with neo4j_driver() as driver, driver.session() as s:
        s.run("MATCH (n) DETACH DELETE n").consume()
        for label in sorted(NEO4J_NODE_LABELS):
            s.run(f"CREATE CONSTRAINT {label.lower()}_id IF NOT EXISTS FOR (n:{label}) REQUIRE n.id IS UNIQUE").consume()
        n = 0
        groups = {}
        for d in dicts:
            groups.setdefault((d["from_node_type"], d["relationship"], d["to_node_type"]), []).append(d)
        for (fl, rel, tl), items in groups.items():
            q = (f"UNWIND $rows AS row "
                 f"MERGE (a:{fl} {{id: row.from_node_id}}) MERGE (b:{tl} {{id: row.to_node_id}}) "
                 f"MERGE (a)-[r:{rel} {{edge_id: row.edge_id}}]->(b) "
                 f"SET r += row.props")
            for i in range(0, len(items), BATCH_SIZE):
                batch = [{"from_node_id": x["from_node_id"], "to_node_id": x["to_node_id"], "edge_id": x["edge_id"],
                          "props": {k: x[k] for k in EDGE_PROPS if x[k] is not None}} for x in items[i:i+BATCH_SIZE]]
                s.run(q, rows=batch).consume()
            n += len(items)
    print(f"   Loaded {n} edges into Neo4j")

def extract_from_neo4j() -> pd.DataFrame:
    """Extract relationships from Neo4j as a tabular edge list."""
    q = ("MATCH (a)-[r]->(b) RETURN r.edge_id AS edge_id, a.id AS from_node_id, labels(a)[0] AS from_node_type, "
         "b.id AS to_node_id, labels(b)[0] AS to_node_type, type(r) AS relationship, properties(r) AS props")
    with neo4j_driver() as driver:
        recs, _, _ = driver.execute_query(q)
    rows = []
    for r in recs:
        d = {k: r[k] for k in ("edge_id", "from_node_id", "from_node_type", "to_node_id", "to_node_type", "relationship")}
        d.update(r["props"])
        rows.append(d)
    out = pd.DataFrame(rows).reindex(columns=[c for c, _ in EDGES_COLSPEC])
    out["timestamp"] = pd.to_datetime(out["timestamp"])
    return out.sort_values("edge_id").reset_index(drop=True)

print("Neo4j functions defined")


Neo4j functions defined


## Task 3.2: Load Source Systems

Load the CSV data into the operational databases (simulating production environment).

In [17]:
print("="*60)
print("TASK 3: Loading Source Systems")
print("="*60)

# TODO: Load data into each source system

# Load PostgreSQL
print("\n📦 Loading orders into PostgreSQL...")
pg_load_orders(orders_df)

# Load Cassandra
print("\n📦 Loading events into Cassandra...")
cas_load_events(events_df)

# Load Neo4j
print("\n📦 Loading edges into Neo4j...")
neo4j_load_edges(edges_df)

print("\n✅ All source systems loaded!")

TASK 3: Loading Source Systems

📦 Loading orders into PostgreSQL...
   Loaded 2500 orders into raw.orders

📦 Loading events into Cassandra...
   Loaded 2500 events into ecommerce.events

📦 Loading edges into Neo4j...
   Loaded 2500 edges into Neo4j

✅ All source systems loaded!


## Task 3.3: Extract and Transform

Extract data from source systems and transform for Redshift staging.

In [ ]:
print("\n" + "="*60)
print("Extracting from Source Systems")
print("="*60)

# TODO: Extract data from each source system

print("\n📤 Extracting from PostgreSQL...")
orders_extracted = extract_from_pg()
print(f"   Extracted {len(orders_extracted)} orders")

print("\n📤 Extracting from Cassandra...")
events_extracted = extract_from_cas()
print(f"   Extracted {len(events_extracted)} events")

print("\n📤 Extracting from Neo4j...")
edges_extracted = extract_from_neo4j()
print(f"   Extracted {len(edges_extracted)} edges")

# TODO: Conform columns to staging specs
print("\n🔄 Conforming data to staging specifications...")
orders = orders_extracted[[c for c,_ in ORDERS_COLSPEC if c in orders_extracted.columns]].copy()
events = events_extracted[[c for c,_ in EVENTS_COLSPEC if c in events_extracted.columns]].copy()
edges = edges_extracted[[c for c,_ in EDGES_COLSPEC if c in edges_extracted.columns]].copy()

print("\n✅ Task 3 Complete - Data extracted and transformed!")

In [ ]:
# ========= Round-trip check: source CSV vs. data extracted back from each system
def roundtrip_check(name, csv_df, extracted, colspec, key):
    cols = [c for c, _ in colspec]
    a = csv_df[cols].copy(); b = extracted[cols].copy()
    for c, k in colspec:
        if k == 'ts':
            a[c] = pd.to_datetime(a[c]); b[c] = pd.to_datetime(b[c])
        elif k == 'b':
            a[c] = a[c].map(lambda v: v if pd.isna(v) else (v if isinstance(v, bool) else str(v) == 'True')).astype('boolean')
            b[c] = b[c].astype('boolean')
        elif k in ('i', 'f'):
            a[c] = pd.to_numeric(a[c]).astype('float64'); b[c] = pd.to_numeric(b[c]).astype('float64')
        else:
            a[c] = a[c].astype('object'); b[c] = b[c].astype('object')
    a = a.sort_values(key).reset_index(drop=True); b = b.sort_values(key).reset_index(drop=True)
    problems = []
    if len(a) != len(b): problems.append(f"row count {len(a)} vs {len(b)}")
    elif not a[key].equals(b[key]): problems.append("keys differ")
    else:
        for c, k in colspec:
            same = (a[c].eq(b[c]) | (a[c].isna() & b[c].isna())) if k != 'f' else \
                   ((a[c] - b[c]).abs().lt(1e-6) | (a[c].isna() & b[c].isna()))
            bad = int((~same.fillna(False)).sum())
            if bad: problems.append(f"{c}: {bad} mismatches")
    print(f"{'OK  ' if not problems else 'FAIL'} {name}: {len(a)} rows, {len(cols)} columns" + ("" if not problems else " -> " + "; ".join(problems)))
    return not problems

ok = all([
    roundtrip_check("orders (Postgres)", orders_df, orders_extracted, ORDERS_COLSPEC, "order_id"),
    roundtrip_check("events (Cassandra)", events_df, events_extracted, EVENTS_COLSPEC, "event_id"),
    roundtrip_check("edges (Neo4j)", edges_df, edges_extracted, EDGES_COLSPEC, "edge_id"),
])
assert ok, "Round-trip mismatch - see output above"


---
# Task 4: Load Data into Redshift

In this task, you will:
- Execute the DDL to create staging, dimension, and fact tables
- Load data into Redshift staging tables
- Populate dimension tables from staging data
- Populate fact tables with dimension key lookups
- Validate successful loading

**Deliverables:**
- Working Redshift connection and execution functions
- Loaded staging, dimension, and fact tables
- Row count validation

## Task 4.1: Define Redshift Functions

In [ ]:
# ========= Redshift Functions =========

session_boto = boto3.Session(region_name=AWS_REGION)
rsd = session_boto.client("redshift-data", region_name=AWS_REGION)

def _rs_kwargs() -> Dict[str, Any]:
    """Build Redshift Data API connection parameters."""
    base = dict(Database=REDSHIFT_DATABASE)
    if REDSHIFT_WORKGROUP:
        base["WorkgroupName"] = REDSHIFT_WORKGROUP
        if REDSHIFT_SECRET_ARN:
            base["SecretArn"] = REDSHIFT_SECRET_ARN
    elif REDSHIFT_CLUSTER_IDENTIFIER and REDSHIFT_DB_USER:
        base["ClusterIdentifier"] = REDSHIFT_CLUSTER_IDENTIFIER
        base["DbUser"] = REDSHIFT_DB_USER
    else:
        raise RuntimeError("Configure Redshift serverless OR provisioned for Data API.")
    return base

def rs_exec(sql: str, return_results=False, timeout_s=900):
    """Execute SQL on Redshift via Data API."""
    # TODO: Implement the following steps:
    # 1. Execute statement using rsd.execute_statement()
    # 2. Poll for completion using rsd.describe_statement()
    # 3. If return_results, fetch using rsd.get_statement_result()
    # 4. Return results as list of dicts
    pass

def rs_batch_insert(table: str, colspec: List[Tuple[str,str]], df: pd.DataFrame):
    """Load DataFrame into Redshift using batch INSERT statements."""
    # TODO: Implement the following steps:
    # 1. Format values based on column types (s, ts, i, f, b)
    # 2. Build multi-row INSERT statements
    # 3. Execute in batches and report progress
    pass

print("Redshift functions defined (implement TODOs)")


## Task 4.2: Execute DDL and Create Tables

In [ ]:
print("="*60)
print("TASK 4: Loading Data into Redshift")
print("="*60)

print("\n📋 Step 1: Executing DDL to create tables...")

# TODO: Read and execute DDL from the markdown file
# 1. Read DDL_MD_PATH file
# 2. Extract SQL blocks from markdown code fences
# 3. Execute each statement (skip CREATE SCHEMA, rewrite schema references)

# Hint: Use regex to extract SQL blocks:
# blocks = re.findall(r"```sql(.*?)```", md_content, flags=re.DOTALL|re.IGNORECASE)

# Hint: Rewrite schema references for public schema:
# s = re.sub(r'\bstg\.', 'public.stg_', s)
# s = re.sub(r'\bdw\.', 'public.dw_', s)


## Task 4.3: Load Staging Tables

In [ ]:
print("\n📦 Step 2: Loading staging tables...")

# TODO: Load each DataFrame into its staging table

# print("\n  Loading orders into stg_orders_raw...")
# rs_batch_insert("public.stg_orders_raw", ORDERS_COLSPEC, orders)

# print("\n  Loading events into stg_events_raw...")
# rs_batch_insert("public.stg_events_raw", EVENTS_COLSPEC, events)

# print("\n  Loading edges into stg_edges_raw...")
# rs_batch_insert("public.stg_edges_raw", EDGES_COLSPEC, edges)

print("\n✅ Staging tables loaded!")

## Task 4.4: Populate Dimension Tables

In [ ]:
print("\n📊 Step 3: Populating dimension tables...")

# TODO: Populate each dimension table from staging data

# Example for dim_date:
# rs_exec("""
#     INSERT INTO public.dw_dim_date (date_key, date_actual, year, quarter, month, day, week_of_year, day_of_week, is_weekend)
#     SELECT DISTINCT
#         CAST(to_char(dt, 'YYYYMMDD') AS INTEGER) AS date_key,
#         dt AS date_actual,
#         EXTRACT(YEAR FROM dt)::SMALLINT,
#         EXTRACT(QUARTER FROM dt)::SMALLINT,
#         EXTRACT(MONTH FROM dt)::SMALLINT,
#         EXTRACT(DAY FROM dt)::SMALLINT,
#         EXTRACT(WEEK FROM dt)::SMALLINT,
#         EXTRACT(DOW FROM dt)::SMALLINT,
#         (EXTRACT(DOW FROM dt) IN (0,6))::BOOLEAN
#     FROM (
#         SELECT order_datetime::date AS dt FROM public.stg_orders_raw WHERE order_datetime IS NOT NULL
#         UNION SELECT ship_datetime::date FROM public.stg_orders_raw WHERE ship_datetime IS NOT NULL
#         UNION SELECT event_ts::date FROM public.stg_events_raw WHERE event_ts IS NOT NULL
#     ) dates WHERE dt IS NOT NULL;
# """)

# TODO: Populate remaining dimensions:
# - dim_customer (from stg_events_raw and stg_orders_raw)
# - dim_product (from stg_events_raw and stg_edges_raw)
# - dim_channel, dim_device, dim_browser, dim_shipping_method, dim_payment_method, dim_campaign
# - dim_os, dim_referrer, dim_ab_variant

print("\n✅ All dimension tables populated!")

## Task 4.5: Populate Fact Tables

In [ ]:
print("\n📊 Step 4: Populating fact tables...")

# TODO: Populate fact tables by joining staging data with dimension surrogate keys

# Example structure for fact_orders:
# rs_exec("""
#     INSERT INTO public.dw_fact_orders (
#         order_id, customer_sk, order_date_key, ship_date_key, channel_sk, ...
#     )
#     SELECT
#         o.order_id,
#         dc.customer_sk,
#         CAST(to_char(o.order_datetime::date, 'YYYYMMDD') AS INTEGER),
#         ...
#     FROM public.stg_orders_raw o
#     LEFT JOIN public.dw_dim_customer dc ON dc.customer_id = o.customer_id AND dc.is_current = TRUE
#     LEFT JOIN public.dw_dim_channel ch ON ch.channel = o.channel
#     ...
# """)

# TODO: Populate:
# - dw_fact_orders
# - dw_fact_events
# - dw_fact_graph_edges

print("\n✅ Task 4 Complete - All data loaded into Redshift!")

---
# Task 5: Optimize Performance and Build OLAP Structures

In this task, you will:
- Verify distribution styles and sort keys are applied
- Run ANALYZE to update statistics
- Create materialized views for common queries

**Deliverables:**
- At least one materialized view for common analytics
- ANALYZE run on key tables

In [ ]:
print("="*60)
print("TASK 5: Optimize Performance")
print("="*60)

# TODO: Create materialized view for daily revenue
print("\n📊 Creating materialized view for daily revenue...")

# Note: Redshift doesn't support "IF NOT EXISTS" for materialized views
# Use DROP + CREATE pattern:
# rs_exec("DROP MATERIALIZED VIEW IF EXISTS public.dw_mv_daily_revenue;")
# rs_exec("""
#     CREATE MATERIALIZED VIEW public.dw_mv_daily_revenue AS
#     SELECT
#         order_date_key,
#         COUNT(*) AS orders,
#         SUM(order_total_usd) AS revenue_usd,
#         AVG(order_total_usd) AS avg_order_value
#     FROM public.dw_fact_orders
#     GROUP BY order_date_key;
# """)


In [ ]:
# TODO: Run ANALYZE on key tables
print("\n📊 Running ANALYZE on tables...")

# for table in ['dw_fact_orders', 'dw_fact_events', 'dw_fact_graph_edges', 
#               'dw_dim_customer', 'dw_dim_product', 'dw_dim_date']:
#     rs_exec(f"ANALYZE public.{table};")
#     print(f"  ✓ ANALYZE complete: {table}")

print("\n✅ Task 5 Complete - Performance optimization done!")

---
# Task 6: Validate and Report Your Results

In this task, you will:
- Run data quality checks
- Execute sample analytical queries
- Generate the final report

**Deliverables:**
- Data quality checks (row counts, null checks)
- Sample analytical query results
- Final report with schema diagram and design rationale

In [ ]:
print("="*60)
print("TASK 6: Validation and Reporting")
print("="*60)

# TODO: Query row counts for all tables
print("\n📊 Row Counts:")
print("-" * 40)

# Example:
# row_counts = rs_exec("""
#     SELECT 'stg_orders_raw' AS table_name, COUNT(*) AS n FROM public.stg_orders_raw
#     UNION ALL SELECT 'stg_events_raw', COUNT(*) FROM public.stg_events_raw
#     UNION ALL SELECT 'dw_fact_orders', COUNT(*) FROM public.dw_fact_orders
#     ...
# """, return_results=True)
# 
# for r in row_counts:
#     print(f"  {r['table_name']}: {r['n']:,}")


In [ ]:
# TODO: Run sample analytical queries
print("\n📊 Sample Analytics - Daily Revenue:")
print("-" * 40)

# Example query using materialized view:
# daily_rev = rs_exec("""
#     SELECT d.date_actual, mv.revenue_usd, mv.orders, mv.avg_order_value
#     FROM public.dw_mv_daily_revenue mv
#     JOIN public.dw_dim_date d ON d.date_key = mv.order_date_key
#     ORDER BY d.date_actual
#     LIMIT 10;
# """, return_results=True)
# 
# display(pd.DataFrame(daily_rev))


In [ ]:
# TODO: Generate final report
print("\n📄 Generating Final Report...")

# Create a markdown report with:
# - Schema diagram (reference project-mermaid-diagram.md)
# - Design rationale
# - Row counts for all tables
# - Sample query results

report_content = f"""# Data Warehouse Build Report

Generated: {datetime.utcnow().isoformat()}Z

## Schema Overview

### Staging Tables
# TODO: Add row counts

### Fact Tables
# TODO: Add row counts

### Dimension Tables
# TODO: Add row counts

## Design Rationale

# TODO: Document your design decisions:
# 1. Why star schema?
# 2. Distribution key choices
# 3. Sort key choices
# 4. Materialized view purpose

## Analytics Capabilities

# TODO: List the types of queries this warehouse supports
"""

# Save report
report_path = os.path.join(BASE_DIR, "warehouse_report.md")
with open(report_path, "w") as f:
    f.write(report_content)

print(f"\n✅ Report saved to: {report_path}")
print("\n" + "="*60)
print("ALL TASKS COMPLETED! ✅")
print("="*60)